# Lab: Build a causal transformer from scratch

**Session 1**

Build a decoder-only language model progressively: one causal head, multi-head attention, a decoder block, then a tiny language model. Every stage is **one to three lines to write**: replace each `____` and run the check cell below it. You finish by training the model and generating text.

## Lab rules

- Use PyTorch primitives only; do not use `nn.MultiheadAttention` or `nn.Transformer`.
- Shapes are explicit: inputs and block outputs use `(B, T, C)`.
- Apply the causal mask to the scores **before** the softmax.
- Each stage has a check cell: pass it before moving on. If you are stuck, set `USE_REFERENCE = True` (see "If you get stuck").

## Tokenization and embeddings

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"  # small: 6 layers, 768 dims, runs on CPU

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)

In [ ]:
def embed(text: str) -> dict:
    """Text in -> tokens + embeddings out."""
    enc = tokenizer(text, return_tensors="pt")
    input_ids = enc["input_ids"]                       # (1, seq_len)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    with torch.no_grad():
        out = model(**enc)

        # 1) Static embeddings: pure table lookup, same vector for a token
        #    whatever the sentence around it.
        static = model.embeddings.word_embeddings(input_ids)[0]      # (seq_len, dim)

        # 2) Position embeddings: one learned vector per position 0..seq_len-1.
        positions = torch.arange(input_ids.size(1)).unsqueeze(0)
        pos = model.embeddings.position_embeddings(positions)[0]     # (seq_len, dim)

    # 3) Contextual embeddings: hidden_states[0] is the input to layer 1
    #    (static + position, then LayerNorm); hidden_states[i] is the output
    #    of layer i.
    layers = [h[0] for h in out.hidden_states]         # list of (seq_len, dim)

    return {
        "tokens": tokens,
        "ids": input_ids[0].tolist(),
        "static": static,
        "position": pos,
        "layers": layers,
        "final": out.last_hidden_state[0],
    }


def show(text: str) -> None:
    r = embed(text)
    print(f"Text: {text!r}")
    print(f"Number of tokens: {len(r['tokens'])}")
    print(f"Embedding dimension: {r['static'].shape[1]}")
    print(f"Number of hidden-state tensors (embeddings + layers): {len(r['layers'])}\n")
    print(f"{'pos':>3}  {'token':<12} {'id':>6}  first 4 dims of final embedding")
    for i, (tok, tid) in enumerate(zip(r["tokens"], r["ids"])):
        vec = r["final"][i][:4].tolist()
        print(f"{i:>3}  {tok:<12} {tid:>6}  {[round(v, 3) for v in vec]}")


def word_vector(text: str, word: str, layer: int = -1) -> torch.Tensor:
    """Embedding of the first occurrence of `word` in `text`, at a given layer."""
    r = embed(text)
    idx = r["tokens"].index(word)
    return r["layers"][layer][idx]

### Part 1: Tokenization

In [ ]:
show("Transformers are unbelievably powerful!")

In [ ]:
for text in ["unbelievably", "teh cat", "12345678", "I love 🍕", "Le chat dort sur le canapé."]:
    print(f"{text!r:<32} -> {tokenizer.tokenize(text)}")

**Question.** Look at the rare word, the typo, the number, the emoji and the French sentence. What happens to each? Why does the `##` prefix exist?

### Part 2: static vs contextual

In [ ]:
s1 = "I deposited money at the bank."
s2 = "We had a picnic on the river bank."
s3 = "I work as an assistant to the regional bank manager"

r1, r2, r3 = embed(s1), embed(s2), embed(s3)
static_bank = r1["static"][r1["tokens"].index("bank")]
static_bank2 = r2["static"][r2["tokens"].index("bank")]
static_bank3 = r3["static"][r3["tokens"].index("bank")]
ctx_bank = word_vector(s1, "bank", layer=-1)
ctx_bank2 = word_vector(s2, "bank", layer=-1)
ctx_bank3 = word_vector(s3, "bank", layer=-1)

cos = lambda a, b: F.cosine_similarity(a, b, dim=0).item()
print("\nCosine similarity between the two 'bank' vectors")
print(f"  static     : {cos(static_bank, static_bank2):.3f}  (should be exactly 1.0)")
print(f"  contextual : {cos(ctx_bank, ctx_bank2):.3f}  (< 1.0: context changed the vector)")
print(f"  contextual : {cos(ctx_bank, ctx_bank3):.3f}  (< 1.0: should be higher than with s2)")

In [ ]:
# cos(bank in s1, bank in s2) at each layer: 0 is the input embedding, 6 the last layer
for layer in range(7):
    sim = cos(word_vector(s1, "bank", layer=layer), word_vector(s2, "bank", layer=layer))
    print(f"layer {layer}: {sim:.3f}")

**Question.** At which layer does the similarity between the two meanings of "bank" drop, and why is it not exactly 1.0 at layer 0?

## Transformer architecture

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
DEVICE = "cpu"  # the model is tiny: a CPU is enough

C, H, MAX_T, V = 32, 4, 16, tokenizer.vocab_size
D = C // H  # size of one head
print({"device": DEVICE, "C": C, "H": H, "D": D, "MAX_T": MAX_T, "V": V})

### Toy corpus

Six tiny sentences: enough to train the model quickly and to see it learn.

In [ ]:
toy_sentences = [
    "the cat sat .", "the cat ate fish .",
    "the dog sat .", "the dog ate food .",
    "the bird flew .", "the bird ate seeds .",
]

def encode(sentence):
    return tokenizer(sentence)["input_ids"]  # [CLS] ... [SEP]

def decode(ids):
    return " ".join(tokenizer.convert_ids_to_tokens(ids))

examples = [encode(s) for s in toy_sentences]
for sentence, ids in zip(toy_sentences, examples):
    print(f"{sentence:<22} -> {ids}")

### Helpers (provided)

- `causal_mask(T, device)`: the lower-triangular boolean mask, `True` where a position may look.
- `check_causal`: changes the future inputs and verifies that the earlier outputs do not move.

In [ ]:
def causal_mask(T, device):
    """Boolean (T, T) matrix: True where position i may look at position j (j <= i)."""
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))


def check_causal(module, x, cutoff, name):
    module.eval()
    with torch.no_grad():
        x_changed = x.clone()
        x_changed[:, cutoff:, :] = torch.randn_like(x_changed[:, cutoff:, :])
        y_a, y_b = module(x), module(x_changed)
    torch.testing.assert_close(y_a[:, :cutoff], y_b[:, :cutoff], atol=1e-5, rtol=1e-5)
    print(f"OK: {name}: positions before {cutoff} ignore future inputs")

### If you get stuck

Every stage has a working version in `reference.py`, next to this notebook. When a stage blocks you after a real attempt, set `USE_REFERENCE = True` in the cell below and run the notebook again from the top: each stage then loads the reference version and the next stages still work.

In [ ]:
USE_REFERENCE = False  # True: every stage below is replaced by its version from reference.py

## Stage 1: one causal attention head

A head turns `(B, T, C)` into `(B, T, D)` in four steps: project to Q, K, V; score with `Q K^T / sqrt(D)`; mask the future; softmax and mix the values. The projections, the scores and the softmax are provided. You write one line: the causal mask.

In [ ]:
class CausalAttentionHead(nn.Module):
    def __init__(self, C, D):
        super().__init__()
        self.D = D
        self.query = nn.Linear(C, D, bias=False)
        self.key = nn.Linear(C, D, bias=False)
        self.value = nn.Linear(C, D, bias=False)

    def forward(self, x):
        # x: (B, T, C) -> (B, T, D)
        T = x.size(1)
        q, k, v = self.query(x), self.key(x), self.value(x)  # each (B, T, D)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.D)  # (B, T, T)
        # TODO: hide the future BEFORE the softmax. Hint: ~causal_mask(T, x.device) is True on the future
        scores = scores.masked_fill(____, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        return weights @ v


if USE_REFERENCE:
    from reference import CausalAttentionHead

In [ ]:
head = CausalAttentionHead(C, D).to(DEVICE)
x = torch.randn(2, 6, C, device=DEVICE)
y = head(x)
assert y.shape == (2, 6, D), f"expected (2, 6, {D}), got {tuple(y.shape)}"
assert torch.isfinite(y).all(), "NaN or inf in the output (mask applied after softmax?)"
print(f"OK: output shape {tuple(y.shape)}")
check_causal(head, x, cutoff=3, name="CausalAttentionHead")

## Stage 2: multi-head attention

Run `H` heads in parallel, concatenate their `(B, T, D)` outputs on the last dimension to get `(B, T, C)`, then apply one output projection. The heads and the projection are created for you: write the one line that joins them.

In [ ]:
class MultiHeadCausalAttention(nn.Module):
    def __init__(self, C, H):
        super().__init__()
        assert C % H == 0
        self.heads = nn.ModuleList([CausalAttentionHead(C, C // H) for _ in range(H)])
        self.out_proj = nn.Linear(C, C, bias=True)

    def forward(self, x):
        # TODO: concatenate the outputs of all heads on the last dim. Hint: torch.cat([...], dim=-1)
        joined = ____  # (B, T, C)
        return self.out_proj(joined)


if USE_REFERENCE:
    from reference import MultiHeadCausalAttention

In [ ]:
multi = MultiHeadCausalAttention(C, H).to(DEVICE)
x = torch.randn(2, 7, C, device=DEVICE)
assert multi(x).shape == x.shape
print(f"OK: output shape {tuple(multi(x).shape)}")
check_causal(multi, x, cutoff=4, name="MultiHeadCausalAttention")

## Stage 3: decoder block

Pre-normalisation and residual connections:

    x = x + attention(LayerNorm(x))
    x = x + MLP(LayerNorm(x))

The MLP (`Linear(C, 4C)`, GELU, `Linear(4C, C)`) and the layers are provided. The attention line is provided. You write the MLP line.

In [ ]:
class MLP(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(C, 4 * C), nn.GELU(), nn.Linear(4 * C, C))

    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, C, H):
        super().__init__()
        self.ln1 = nn.LayerNorm(C)
        self.attn = MultiHeadCausalAttention(C, H)
        self.ln2 = nn.LayerNorm(C)
        self.mlp = MLP(C)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        # TODO: MLP sub-layer, same pattern. Hint: x + self.mlp(self.ln2(x))
        x = ____
        return x


if USE_REFERENCE:
    from reference import MLP, DecoderBlock

In [ ]:
block = DecoderBlock(C, H).to(DEVICE)
x = torch.randn(2, 8, C, device=DEVICE)
assert block(x).shape == x.shape
print(f"OK: output shape {tuple(block(x).shape)}")
check_causal(block, x, cutoff=4, name="DecoderBlock")

## Stage 4: tiny decoder language model

Token embedding + position embedding, one decoder block, a final LayerNorm, and a linear head to the vocabulary. With targets, the model also returns the cross-entropy loss. Everything is provided: read `forward`, then run the check.

In [ ]:
class TinyDecoderLM(nn.Module):
    def __init__(self, V, C, H, max_t):
        super().__init__()
        self.max_t = max_t
        self.token_embedding = nn.Embedding(V, C)
        self.position_embedding = nn.Embedding(max_t, C)
        self.block = DecoderBlock(C, H)
        self.ln_f = nn.LayerNorm(C)
        self.lm_head = nn.Linear(C, V, bias=True)

    def forward(self, token_ids, targets=None):
        # token_ids: (B, T), T <= max_t
        T = token_ids.size(1)
        assert T <= self.max_t, f"Sequence length {T} exceeds max_t={self.max_t}"
        positions = torch.arange(T, device=token_ids.device)  # (T,)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)  # (B, T, C)
        logits = self.lm_head(self.ln_f(self.block(x)))  # (B, T, V)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss


if USE_REFERENCE:
    from reference import TinyDecoderLM

In [ ]:
model = TinyDecoderLM(V, C, H, MAX_T).to(DEVICE)
ids = torch.tensor([examples[0]], dtype=torch.long, device=DEVICE)
logits, loss = model(ids, targets=ids)
assert logits.shape == (1, ids.size(1), V)
assert loss.ndim == 0 and torch.isfinite(loss)
print(f"OK: logits {tuple(logits.shape)}, initial loss {loss.item():.2f} (random guess is ln V = {math.log(V):.2f})")

## Train and generate (provided)

The model learns to predict the next token: the input is a sentence without its last token, the target is the same sentence shifted by one.

    input:  [CLS] the cat sat .
    target: the   cat sat .   [SEP]

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
data = [torch.tensor([ids], dtype=torch.long, device=DEVICE) for ids in examples]

for epoch in range(150):
    model.train()
    total = 0.0
    for ids in data:
        _, loss = model(ids[:, :-1], targets=ids[:, 1:])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    if epoch % 30 == 0 or epoch == 149:
        print(f"epoch {epoch:3d}  mean loss {total / len(data):.3f}")

In [ ]:
@torch.no_grad()
def generate(prompt, max_new=8, temperature=0.0):
    """Append tokens one at a time; temperature 0 means always the most likely token."""
    model.eval()
    ids = torch.tensor([encode(prompt)[:-1]], dtype=torch.long, device=DEVICE)  # drop [SEP]
    for _ in range(max_new):
        logits, _ = model(ids[:, -MAX_T:])
        last = logits[0, -1]
        if temperature == 0:
            next_id = last.argmax()
        else:
            next_id = torch.multinomial(torch.softmax(last / temperature, dim=-1), 1)[0]
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
        if next_id.item() == tokenizer.sep_token_id:
            break
    return decode(ids[0].tolist()[1:])

for prompt in ["the cat", "the dog", "the bird"]:
    print(f"{prompt!r:>12} -> {generate(prompt)}")

## Look inside: from text to next-token probabilities

    text -> token IDs -> token + position embeddings -> decoder block -> logits -> softmax

In [ ]:
prompt = "the cat"
prompt_ids = torch.tensor([encode(prompt)[:-1]], dtype=torch.long, device=DEVICE)
print("tokens:", decode(prompt_ids[0].tolist()), " ids:", prompt_ids.tolist())

model.eval()
with torch.no_grad():
    logits, _ = model(prompt_ids)
    probabilities = torch.softmax(logits[:, -1, :], dim=-1)

top_p, top_ids = probabilities.topk(k=5, dim=-1)
for p, token_id in zip(top_p[0], top_ids[0]):
    print(f"{decode([token_id.item()]):>8}: {p.item():.1%}")

## Questions

Run the cells, then answer each question in one sentence.

1. After training, after `the cat` the model gives roughly 50% to `sat` and 50% to `ate`. Why can it never be sure? (Look at the corpus.)
2. Run the next cell: `generate("the", temperature=0)` several times, then with `temperature=1.5`. What changes, and which words can follow `the`?

In [ ]:
for t in [0.0, 0.0, 1.5, 1.5, 1.5]:
    print(f"temperature {t}: {generate('the', temperature=t)}")

3. Run the next cell. It applies the mask **after** the softmax instead of before (it sets the future weights to 0). Does `check_causal` still pass? Do the rows of weights still sum to 1?

In [ ]:
class WrongMaskHead(CausalAttentionHead):
    """Same head, but the mask is applied AFTER the softmax."""

    def forward(self, x):
        T = x.size(1)
        q, k, v = self.query(x), self.key(x), self.value(x)
        weights = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.D), dim=-1)
        weights = weights * causal_mask(T, x.device)  # zero the future after the softmax
        self.last_weights = weights
        return weights @ v


wrong = WrongMaskHead(C, D).to(DEVICE)
x = torch.randn(2, 6, C, device=DEVICE)
wrong(x)
print("row sums of the weights:", [round(s, 2) for s in wrong.last_weights[0].sum(dim=-1).tolist()])
try:
    check_causal(wrong, x, cutoff=3, name="WrongMaskHead")
except AssertionError:
    print("check_causal FAILS: the future still leaks through the softmax normaliser")

4. Run the next cell: a sequence longer than `MAX_T`. What error do you get, and why does the model have a maximum length?

In [ ]:
too_long = torch.tensor([examples[0] * 4], dtype=torch.long, device=DEVICE)  # longer than MAX_T
print("sequence length:", too_long.size(1), "MAX_T:", MAX_T)
try:
    model(too_long)
except AssertionError as e:
    print("AssertionError:", e)

## Submission checklist

- Head, multi-head, block and language model all pass their cells.
- The loss decreases during training and `generate("the cat")` returns a plausible sentence.
- You can explain why the mask is applied before the softmax, and why masking after it is incorrect.